In [1]:
# PART 3: ML Model (Azure ML)
# Task 3: Train Model

In [22]:
# Steps:
# Upload CSV dataset to Azure ML Workspace

# Use:
# Designer OR Notebook

# Train:
# Classification model (Fraud Detection)

# 🎯 Student Task:
# Identify important features

# Evaluate:
# Accuracy
# Precision

In [2]:
import pandas as pd

df = pd.read_csv("insurance_claims_large.csv")
df.head()

,Customer_ID,Claim_Amount,Claim_Type,Location,Previous_Claims,Fraud
0,C0001,86834,Vehicle,Delhi,4,Yes
1,C0002,17654,Health,Delhi,2,No
2,C0003,102164,Vehicle,Pune,0,No
3,C0004,143454,Vehicle,Bangalore,0,No
4,C0005,10929,Health,Delhi,0,No


In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["Claim_Type"] = le.fit_transform(df["Claim_Type"])
df["Location"] = le.fit_transform(df["Location"])
df["Fraud"] = le.fit_transform(df["Fraud"])   # Yes=1, No=0

In [4]:
X = df[["Claim_Amount", "Previous_Claims", "Location", "Claim_Type"]]
y = df["Fraud"]

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [7]:
y_pred = model.predict(X_test)

In [8]:
from sklearn.metrics import accuracy_score, precision_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)

print("\nDetailed Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 1.0
Precision: 1.0

Detailed Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        53
           1       1.00      1.00      1.00        47

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



In [9]:
import pandas as pd

importance = model.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importance
}).sort_values(by="Importance", ascending=False)

print(feature_importance)

           Feature  Importance
0     Claim_Amount    0.536642
1  Previous_Claims    0.452018
2         Location    0.007161
3       Claim_Type    0.004178


In [13]:
# PART 4: Model Deployment (API)
# ✅ Task 4: Deploy Model as Endpoint

In [14]:
# Steps:
# Register Model

# Create:

# Inference script (score.py)

# Environment

# Deploy → Real-time endpoint

In [15]:
import json
import joblib
import numpy as np

def init():
    global model
    model = joblib.load("fraud_model.pkl")

def run(data):
    try:
        input_data = data["data"]
        results = []

        # same encoding as training
        location_map = {
            "Delhi": 0,
            "Mumbai": 1,
            "Bangalore": 2,
            "Chennai": 3,
            "Pune": 4,
            "Kolkata": 5
        }

        for item in input_data:
            features = [[
                item["Claim_Amount"],
                item["Previous_Claims"],
                location_map.get(item["Location"], 0),
                0   # Claim_Type default
            ]]

            prediction = model.predict(features)[0]

            results.append({
                "Fraud_Prediction": int(prediction)
            })

        return {"result": results}

    except Exception as e:
        return {"error": str(e)}

In [19]:
from dotenv import load_dotenv
import os

# Load env variables
load_dotenv(dotenv_path=r"C:\Users\naina\Capgemini\Generative Ai\.env")

key = os.getenv("Deploy_api")
url = os.getenv("Deploy_endpoint")

In [20]:
import requests
import json


headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {key}"
}

data = {
    "Inputs": {
        "input1": [
            {
                "Customer_ID": "C001",
                "Claim_Amount": 70000,
                "Claim_Type": "Vehicle",
                "Location": "Delhi",
                "Previous_Claims": 3
            }
        ]
    }
}

response = requests.post(url, headers=headers, json=data)

print("Status:", response.status_code)
print(response.json())

Status: 200
{'Results': {'WebServiceOutput0': [{'Customer_ID': 'C001', 'Claim_Amount': 70000, 'Claim_Type': 'Vehicle', 'Location': 'Delhi', 'Previous_Claims': 3, 'Scored Labels': False, 'Scored Probabilities': 0.41346096392843495}]}}


In [21]:
result = response.json()

output = result["Results"]["WebServiceOutput0"][0]

print("Fraud:", output["Scored Labels"])
print("Probability:", output["Scored Probabilities"])

Fraud: False
Probability: 0.41346096392843495
